# 00 · 论文身份、复现边界与公式审计

**任务目标**：复现用户提供 PDF 的实际正文，而非根据文件名猜测内容。所有验证均在 CUDA Ubuntu 容器中、通过 uv 项目级 `.venv` 执行；核心实验是 CPU 统计计算，不需要 GPU。

| 项目 | 核对结果 |
|---|---|
| 当前正确文件名 | 2024_Metrics for Dataset Demographic Bias - A Case Study on Facial Expression Recognition.pdf |
| 原始文件名（已更正） | 2024_CS_Metrics and Algorithms for Identifying and Mitigating Bias in AI Design- A Counterfactual Fairness Approach.pdf |
| PDF 正文标题 | Metrics for Dataset Demographic Bias: A Case Study on Facial Expression Recognition |
| 作者 | Iris Dominguez-Catena, Daniel Paternain, Mikel Galar |
| 版本 | arXiv:2303.15889v2，2024-06-25；正文及附录共 45 页 |
| 正确源码 | https://github.com/irisdominguez/Dataset_Bias_Metrics |
| 用户给的连字符仓库 | Dataset-Bias-Metrics，README 指向 2022 年 *Assessing Demographic Bias Transfer…*，不是本篇正文指标库 |

**数据真实性**：20 个 CSV 来自作者 `example_data`，包含打乱行序的 `age/race/gender/label`，不是我们生成的数据。人口属性由 FairFace 估计，不能解释为自报身份。

**范围**：正文 Sec.III–V、Fig.2–8 与附录 B 的统计分析可在公开元数据上重新计算。Sec.IV 的原图预处理/FairFace 标注不重跑；附录 A 只重建列联表和 Fig.9–12(a) 指标曲线，不声称重跑 AffectNet 图像 ResNet50 训练或复现其 recall。改进与合成下游实验均标记为“本组扩展”。

这篇论文研究描述性统计偏差，没有给出保证反事实公平的因果算法。后续的反事实实验是独立扩展，不能替代原论文。

In [1]:
from pathlib import Path
import os, sys, json, warnings, time
import numpy as np
import pandas as pd
from IPython.display import display, Image, Markdown
ROOT = Path.cwd() if (Path.cwd()/"pyproject.toml").exists() else Path.cwd().parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT/"src"))
from experiments import *
warnings.filterwarnings("ignore", category=FutureWarning)
print("项目目录:", ROOT)
print("Python:", sys.version.split()[0], "NumPy:", np.__version__, "Pandas:", pd.__version__)


项目目录: /workspace
Python: 3.11.15 NumPy: 1.26.4 Pandas: 2.1.4


In [2]:
# 对原 PDF 做内容核对，记录 SHA256，避免文件名导致论文串题。
import fitz, hashlib
pdf = ROOT / '2024_Metrics for Dataset Demographic Bias - A Case Study on Facial Expression Recognition.pdf'
doc = fitz.open(pdf)
assert len(doc) == 45
first = doc[0].get_text()
assert "Metrics for Dataset Demographic Bias" in first
print("PDF 页数:", len(doc))
print("PDF SHA256:", hashlib.sha256(pdf.read_bytes()).hexdigest())
print(first[:800])
data = load_data()
provenance_info = provenance(data)
print("公开真实数据集:", len(data), "总记录数:", provenance_info['total_rows'])
print("固定上游 commit:", provenance_info['upstream_commit'])
display(pd.DataFrame([{ 'dataset':n, 'rows':len(d), **{c:d[c].nunique() for c in COMPONENTS}} for n,d in data.items()]))

PDF 页数: 45
PDF SHA256: b31ce26bc7700cb90a1e9126038e25b9e6d6913286e1957f211be430b405fd9c
1
Metrics for Dataset Demographic Bias: A Case
Study on Facial Expression Recognition
Iris Dominguez-Catena , Student Member, IEEE, Daniel Paternain , Member, IEEE,
Mikel Galar , Member, IEEE
Abstract—Demographic biases in source datasets have been
shown as one of the causes of unfairness and discrimination in
the predictions of Machine Learning models. One of the most
prominent types of demographic bias are statistical imbalances
in the representation of demographic groups in the datasets.
In this paper, we study the measurement of these biases by
reviewing the existing metrics, including those that can be
borrowed from other disciplines. We develop a taxonomy for
the classification of these metrics, providing a practical guide
for the selection of appropriate metrics. To illustrate the u


公开真实数据集: 20 总记录数: 794166
固定上游 commit: a70bdc28cc96b338f4d5fdebce5064b56d80de57


,dataset,rows,age,race,gender
0,adfes,132,2,3,2
1,affectnet,286341,9,7,2
2,caer-s,69757,9,7,2
3,ck+,3607,4,7,2
4,ck,2820,3,6,2
5,expw,83208,9,7,2
6,fer2013,33113,9,7,2
7,ferplus,32673,9,7,2
8,gemep,2913,5,1,2
9,isafe,26916,3,3,2


## 1. 指标与统一约定

令 $p_g=n_g/n$，$p_{gy}=n_{gy}/n$，$e_{gy}=p_gp_y$。`0 log 0=0`。Richness 为实际观测组数。先去掉全零边缘，缺失预设组另记覆盖率。

- 表征偏差：$R$、$H=-\sum p_g\log p_g$、$ENS=e^H$、$SEI=H/\log R$、NSD、IR、BP、$D=\sum p_g^2$、$1/D$、$1-D$。ENS/SEI 越大越均衡；D/NSD/IR/BP 越大越偏。
- 全局刻板偏差：Cramér V、Tschuprow T、Pearson C、Theil U（双方向）、NMI。论文 NMI 是 $I(G;Y)/H(G,Y)$，不能直接套 sklearn 默认归一化。
- 局部刻板偏差：NPMI、Ducher Z。正值表示超额出现，负值表示不足；不是分类错误率。

独立实现 `src/metrics.py` 使用 NumPy 概率表；作者原码保留不修改。NumPy 固定 1.26.4，兼容上游 `np.NaN`；Pandas 固定 2.1.4，兼容上游整数表写入小数的历史写法。

In [3]:
# 先用可手算的列联表验证：独立、不平衡但独立、完全关联。
from metrics import representation, association, fixed_universe
independent = np.array([[25,25],[25,25]])
imbalanced_independent = np.array([[45,45],[5,5]])
perfect = np.array([[50,0],[0,50]])
assert np.isclose(association(independent)[0]['V'], 0)
assert np.isclose(association(imbalanced_independent)[0]['V'], 0)
assert np.isclose(association(perfect)[0]['V'], 1)
assert np.isclose(representation([50,50])['ENS'], 2)
assert np.isclose(representation([100])['ENS'], 1)
assert np.isnan(representation([100])['SEI'])
# 单组 V 不可识别：NaN 不是“没有偏差”。
assert np.isnan(association([[50,50]])[0]['V'])
print("手算边界案例全部通过。")

手算边界案例全部通过。


## 2. 公式、源码与解释之间的差异

1. **Eq.(12)** 的 PDF 将 Pearson C 写成 $\sqrt{(\chi^2/n)/(1-\chi^2/n)}$。作者调用 SciPy 的定义为 $\sqrt{(\chi^2/n)/(1+\chi^2/n)}$。完全关联的 2×2 表中前者分母为 0；标准值为 $\sqrt{1/2}$。这是可直接证明的公式排版/定义差异，不是实验发现“算法必然失效”。
2. **Eq.(13)** 分子 $H(P_1)-H(P_1\mid P_2)=I$，但分母印为 $H(P_2)$；作者 `theilsu(x,y)` 用 $H(x)$。我们记录双方向并遵循作者实现进行主结果比较。
3. **Table I** 将 Shannon H 与 bias 关系列为 direct，与正文“熵越高越多样”相矛盾；应按正文解释。NSD 的表中区间也不包括其可取的 0。
4. **`chisq` 函数** 返回 p 值，不是 docstring 声称的统计量；不属于本文最终推荐指标。
5. **多组件 richness**：作者对每列 `nunique` 相乘，得到理论笛卡尔积大小，未必等于实际出现的交叉群体数。单组件主实验不受此问题影响。
6. **缺失群体**：只存在一个组时 IR=1；SEI 未定义。不能把“已出现组内部均衡”读成所有群体都得到覆盖。

以上与作者原码逐项对应，后面的 Notebook 使用固定定义，避免悄悄修复后宣称完全一致。

In [4]:
audit = counterexamples()
display(audit)
assert audit['joint_observed_groups'] == 2
assert audit['upstream_joint_R'] == 4
assert np.isclose(audit['pearson_perfect_standard'], np.sqrt(.5))
assert audit['single_group']['coverage'] == .5
print("审计反例已保存 results/tables/counterexamples.json")

{'single_group': {'R': 1,
  'ENS': 1.0,
  'D': 1.0,
  '1/D': 1.0,
  '1-D': 0.0,
  'H': -0.0,
  'SEI': nan,
  'NSD': nan,
  'IR': 1.0,
  'BP': 1.0,
  'coverage': 0.5,
  'H_over_logK': -0.0,
  'ENS_over_K': 0.5},
 'marginal_gender_ENS': 2.0,
 'marginal_race_ENS': 2.0,
 'joint_observed_groups': 2,
 'joint_fixed_groups': 4,
 'upstream_joint_R': 4,
 'pearson_perfect_standard': 0.7071067811865476,
 'pearson_paper_eq12_denominator': 0,
 'chisq_upstream_output': 1.1258564622753109e-22}

审计反例已保存 results/tables/counterexamples.json


## 3. 完整性与局限

- 20 数据集元数据全量计算，不抽样代替主结果；各 CSV 校验和存于 `results/provenance.json`。
- 观测偏差不等价于模型歧视或因果效应。未获得同一人身份、图片特征和模型输出，不能从这些 CSV 推断真实模型公平性。
- 原图、FairFace 自动标注的误差、同一主体重复帧、被剔除的人脸，均可能影响统计结论；公开元数据不含主体标识，无法修复聚类依赖。
- 原论文已讨论自动标注局限和“数据偏差不线性传递”。本组贡献是定量审计、边界反例和可复现的缓解探索，而非把这些已知局限包装成首次发现。

来源：本地 PDF Sec.III、IV、V、Appendix A/B；作者 README / 源码；https://arxiv.org/abs/2303.15889v2 。